# Bunch fitting for the NC-CC mapping used for boosting purity of NC events
This script is used to extract the center positions of the CC events. The script is adapted from Yue's original fitting implementation, and follows from: https://github.com/S81D/BNB_bunches

We don't need to include off-beam here because:
1. It is a continuous in time (CIT) component to our spill structure, and since
2. We have plenty of statistics, we can easily fit the spill structure on top of the off-beam CIT, and not influence our fitting results

In [ ]:
''' Selection criteria for CCinc sample (same as the CC/NC bunch comparisons for MC!)
    - No Veto activity (not a dirt muon)
    - Prompt cluster (cluster index = 0)
    - CCB < 0.2 (charge distribution is focused, with lots of illumination in the tank)
    - Brighest cluster
    - (not charge enforcement, the distribution is pretty broad, and wasn't used in the MC comparison)
    - beam-correlated, cluster time within spill window
    - greater than 100 PMT hits (enforce the muon does not clip the tank)
'''

from lib.constants import *
from lib.imports import *
import warnings
from scipy.optimize import OptimizeWarning

def CCinc(T1,TMRD1,NV1,B1,CCB1,CH1,CT,CN1):
    
    # No MRD selection cuts are applied to insulate us from MRD dead paddles / systematics
    
    if(NV1==0):   # NoVeto
        return False
    if(B1==0):    # Brightest
        return False
    if(CCB1>0.2 or CCB1<0): # Cluster Charge Balance < 0.2
        return False
    if(CH1<100):   # indicative the muon topology is not center-aligned
        return False
    if(CT>1800 or CT<150):  # inside spill window (adjusted!!!)
        return False
        
    # 3. Prompt (first) cluster
    if CN1 != 0:
        return False
    
    # if it passes
    return True


def extract_run_number(file_name):
    return int(file_name.split('_')[0][1:])  # Split by '_' and remove 'R' to get the run number


# when dropping the MRD for bunch fitting, the FY22 and FY23 are remarkably close in their spill structure when filtering out bad runs
# we can therefore combine the runs from separate epochs and just filter out the bad ones
# Bad runs are determined from noise and BRF investigations (see Steven Doran's thesis)

def filter_runs(run):
    
    # empty runs (no BRF fits)
    empty = [3221, 3397, 3567, 3628, 3661, 3792, 4106, 4112, 4116, 4125, 4351, 4357, 4471, 4676, 4725, 5348, 5335, 5345,
             5379, 5389, 5408, 5413, 5434, 5457, 5477, 5489, 5493, 5501, 5576, 5577, 5588, 5613, 5616, 5622, 5623,
             5639, 5641, 5642, 5648, 5650, 5651, 5655, 5657, 5658, 5660, 5661, 5663, 5665, 5668]
    
    # bad FY23 runs
    BL_23 = [4077, 4080, 4081, 4082, 4083, 4084, 4085, 4086, 4087, 4090, 4091, 4092,
             4093, 4095, 4096, 4097, 4098, 4099, 4100, 4102, 4103, 4105, 4115]

    
    if run in empty and run not in BL_23:
        return False
    
    else:
        return True


# -----------------------------------------------------------
# REQUIRES USER INPUT
directory = 'FY22_23/'
csv_file = 'lib/POT_trigger_FY22_23_summary.csv'
plot_dir = 'plots/bunch_timing/'
os.system('mkdir -p ' + plot_dir)

print('done')

In [ ]:
file_names = [file for file in os.listdir(directory) if os.path.isfile(os.path.join(directory, file))]
print('There are: ', len(file_names), ' files\n')

# POT file
pot_df = pd.read_csv(csv_file)

# Ensure RUN is integer
pot_df["RUN"] = pot_df["RUN"].astype(int)

# Create dictionary: {run: POT}
pot_dict = dict(zip(pot_df["RUN"], pot_df["TOR860_POTe12"]))

counter = 0; total_POT_TOR860 = 0
good_run_count = 0; bad_run_count = 0
cluster_time = []; cluster_charge = []

for file_name in file_names:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    
    include_run = filter_runs(run_number)    # filter option (1 = good only, 0 = all)
    
    if include_run == False:
        print('\nNot including Run', run_number)
        continue
        
    print('\nRun: ', run_number, '(', (counter), '/', len(file_names), ')')

    # lookup run number in POT dictionary
    if run_number not in pot_dict:
        print(f"Skipping {file_name}: run {run_number} not found in POT csv file")
        continue

    E_TOR860 = pot_dict[run_number]
    total_POT_TOR860 += E_TOR860
    
    
    with uproot.open(os.path.join(directory, file_name)) as file:
        
        Event = file["data"]
        
        CT2 = Event["cluster_time_BRF"].array()     # BRF-corrected timing
        CPE1 = Event["cluster_PE"].array()
        CCB1 = Event["cluster_Qb"].array()
        CH1 = Event["cluster_Hits"].array()
        B1 = Event["isBrightest"].array()
        TMRD1 = Event["TankMRDCoinc"].array()
        T1 = Event["MRD_Track"].array()
        NV1 = Event["NoVeto"].array()
        CN1 = Event["cluster_Number"].array()
        

        events_per_run = 0
        for i in trange(len(CT2)):
            
            # CC selection
            is_CCinc = CCinc(T1[i],TMRD1[i],NV1[i],B1[i],CCB1[i],CH1[i],CT2[i],CN1[i])
            if(is_CCinc==False):
                continue
            
            cluster_time.append(CT2[i])
            cluster_charge.append(CPE1[i])
            events_per_run += 1

        print(events_per_run, ' events')
        

total_POT_TOR860 = total_POT_TOR860*(1e12)/(1e20)

print('\nAfter selection cuts, we have: ', len(cluster_time), ' CC candidates\n\n')
print('Total POT (e20)')
print('---------------')
print('E_TOR860: ' + str(total_POT_TOR860))

In [ ]:
def gaussian(x, amp, mean, sigma):
    """Gaussian function for fitting"""
    return amp * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))

def fit_gaussian_constrained(x, y, initial_mean, fixed_spacing, bunch_idx):
    """
    Fit gaussian with mean constrained near expected position.
    More lenient bounds for low statistics.
    """
    if len(x) < 3 or np.sum(y) < 5:  # Skip if too few points or events
        return None, None
    
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=OptimizeWarning)
            
            # More generous bounds for low stats
            mean_tolerance = 8  # Allow ±8 ns deviation
            amp_guess = max(y) if max(y) > 0 else 1
            sigma_guess = 2.0  # Start closer to expected
            
            params, pcov = curve_fit(
                gaussian, x, y,
                p0=[amp_guess, initial_mean, sigma_guess],
                bounds=(
                    [0, initial_mean - mean_tolerance, 0.5],  # Lower bounds
                    [amp_guess * 3, initial_mean + mean_tolerance, 10]  # Upper bounds
                ),
                maxfev=10000
            )
            perr = np.sqrt(np.diag(pcov))
            return params, perr
    except (RuntimeError, ValueError) as e:
        return None, None

def compute_fit_quality(x, y, params):
    """
    Compute fit quality metric that works better for low statistics.
    Uses weighted residuals instead of chi-square.
    """
    if params is None:
        return np.inf
    
    y_fit = gaussian(x, *params)
    
    # For low statistics, use weighted sum of squared residuals
    # Add small constant to avoid division by zero
    weights = 1.0 / (np.sqrt(y + 1))  # Poisson-like weighting
    residuals = (y - y_fit) * weights
    
    return np.sum(residuals**2)

def perform_stable_fit(timing_data, id_name, fixed_spacing, first_peak_guess, 
                       SavePath, n_bunches=81, scan_range=20):
    """
    Perform bunch fitting with FIXED spacing, only scanning global offset.
    
    Parameters:
    -----------
    timing_data : array
        Event times
    id_name : str
        Identifier for plots
    fixed_spacing : float
        Known bunch spacing (18.936 ns)
    first_peak_guess : float
        Initial guess for first bunch center
    SavePath : str
        Directory for saving plots
    n_bunches : int
        Number of bunches (default 81)
    scan_range : float
        Range to scan around first_peak_guess (±scan_range ns)
    """
    
    #bins_TSDt = np.arange(150, 1800, 2)
    bins_TSDt = np.arange(185, 1740, 2)
    h_TSDt, _ = np.histogram(timing_data, bins=bins_TSDt)
    bin_centers = (bins_TSDt[:-1] + bins_TSDt[1:]) / 2
    
    # Scan only the global offset
    scan_step = 0.2  # Coarser steps since spacing is fixed
    scan_offsets = np.arange(first_peak_guess - scan_range, 
                             first_peak_guess + scan_range, 
                             scan_step)
    
    best_quality = np.inf
    best_offset = first_peak_guess
    best_results = None
    
    print(f"Scanning {len(scan_offsets)} offset positions with fixed spacing={fixed_spacing} ns")
    
    for offset in trange(len(scan_offsets), desc="Finding best global offset"):
        current_offset = scan_offsets[offset]
        
        bunch_means = []
        bunch_sigmas = []
        bunch_mean_errors = []
        fit_qualities = []
        
        # Fit all bunches with this offset
        for i in range(n_bunches):
            bunch_center = current_offset + i * fixed_spacing
            
            # Define window around expected bunch position
            half_window = fixed_spacing * 0.45  # Use 90% of spacing
            start = bunch_center - half_window
            end = bunch_center + half_window
            
            mask = (bin_centers >= start) & (bin_centers < end)
            x = bin_centers[mask]
            y = h_TSDt[mask]
            
            if len(x) < 3:
                bunch_means.append(np.nan)
                bunch_sigmas.append(np.nan)
                bunch_mean_errors.append(np.nan)
                fit_qualities.append(np.inf)
                continue
            
            params, errors = fit_gaussian_constrained(
                x, y, bunch_center, fixed_spacing, i
            )
            
            if params is None:
                bunch_means.append(np.nan)
                bunch_sigmas.append(np.nan)
                bunch_mean_errors.append(np.nan)
                fit_qualities.append(np.inf)
            else:
                bunch_means.append(params[1])
                bunch_sigmas.append(params[2])
                bunch_mean_errors.append(errors[1])
                fit_qualities.append(compute_fit_quality(x, y, params))
        
        # Total quality for this offset (sum of all bunch fit qualities)
        total_quality = np.nansum(fit_qualities)
        
        if total_quality < best_quality:
            best_quality = total_quality
            best_offset = current_offset
            best_results = {
                'offset': current_offset,
                'means': bunch_means,
                'sigmas': bunch_sigmas,
                'mean_errors': bunch_mean_errors,
                'qualities': fit_qualities
            }
    
    print(f"\nBest global offset: {best_offset:.2f} ns")
    print(f"Best fit quality score: {best_quality:.2f}")
    
    # Filter out failed fits
    valid_means = [m for m in best_results['means'] if not np.isnan(m)]
    valid_sigmas = [s for s in best_results['sigmas'] if not np.isnan(s)]
    
    print(f"Successfully fitted {len(valid_means)}/{n_bunches} bunches")
    print(f"Mean bunch width: {np.mean(valid_sigmas):.3f} ± {np.std(valid_sigmas):.3f} ns")
    
    # Generate plots
    plot_timing_histogram(bins_TSDt, h_TSDt, id_name, SavePath)
    plot_fitted_bunches(bins_TSDt, h_TSDt, best_results, fixed_spacing, 
                        id_name, SavePath)
    plot_bunch_diagnostics(best_results, fixed_spacing, id_name, SavePath)
    
    return best_results

def plot_timing_histogram(bins_TSDt, h_TSDt, id_name, SavePath):
    """Plot basic timing histogram"""
    plt.figure(figsize=(12, 4))
    plt.hist(bins_TSDt[:-1], bins=bins_TSDt, weights=h_TSDt, 
             alpha=0.7, color='tomato', histtype='stepfilled')
    plt.hist(bins_TSDt[:-1], bins=bins_TSDt, weights=h_TSDt, 
             alpha=0.7, color='black', histtype='step', linewidth=2)
    plt.xlabel('cluster time [ns]', fontsize=16, loc='right')
    plt.ylabel('events / 2ns', fontsize=16)
    plt.title(f'Beam Timing Histogram for {id_name}', fontsize=24)
    ax = plt.gca()
    ax.tick_params(axis='both', direction='in')
    ax.xaxis.set_major_locator(MultipleLocator(250))
    ax.xaxis.set_minor_locator(MultipleLocator(50))
    plt.text(0.16, 1.09, f'{total_POT_TOR860:.2f}' + r'$\times 10^{20}$' + ' POT', 
         horizontalalignment='right', verticalalignment='top', 
         transform=plt.gca().transAxes, fontsize=14)
    plt.tight_layout()
    plt.savefig(SavePath + f'TimingHist_{id_name}.png', dpi=300, 
                bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.close()

def plot_fitted_bunches(bins_TSDt, h_TSDt, results, fixed_spacing, 
                        id_name, SavePath):
    """Plot histogram with fitted Gaussians overlaid"""
    bin_centers = (bins_TSDt[:-1] + bins_TSDt[1:]) / 2
    
    plt.figure(figsize=(12, 4))
    plt.hist(bins_TSDt[:-1], bins=bins_TSDt, weights=h_TSDt, 
             alpha=0.7, color='C0')
    
    for i, mean in enumerate(results['means']):
        if np.isnan(mean):
            continue
        
        sigma = results['sigmas'][i]
        half_window = fixed_spacing * 0.45
        start = mean - half_window
        end = mean + half_window
        
        mask = (bin_centers >= start) & (bin_centers < end)
        x = bin_centers[mask]
        y = h_TSDt[mask]
        
        if len(x) < 3:
            continue
        
        # Refit for plotting
        params, _ = fit_gaussian_constrained(x, y, mean, fixed_spacing, i)
        if params is not None:
            x_smooth = np.linspace(start, end, 100)
            y_fit = gaussian(x_smooth, *params)
            plt.plot(x_smooth, y_fit, 'r-', linewidth=1, alpha=0.8)
    
    plt.xlabel('cluster time [ns]', loc='right', fontsize=16)
    plt.ylabel('events / 2ns', fontsize=16)
    plt.title(f'Beam Timing with Gaussian Fits for {id_name}', fontsize=24)
    plt.tight_layout()
    plt.savefig(SavePath + f'TimingHist_gaussian_fitting_{id_name}.png', 
                dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.close()

def plot_bunch_diagnostics(results, fixed_spacing, id_name, SavePath):
    """Plot diagnostic plots for bunch fitting"""
    valid_means = [m for m in results['means'] if not np.isnan(m)]
    valid_sigmas = [s for s in results['sigmas'] if not np.isnan(s)]
    
    # Bunch spacing intervals
    intervals = np.diff([m for m in results['means'] if not np.isnan(m)])
    
    if len(intervals) > 0:
        plt.figure(figsize=(5, 4))
        plt.hist(intervals, range=(15, 25), bins=20, alpha=0.7, color='C0')
        plt.axvline(x=fixed_spacing, color='r', linestyle='--', linewidth=2,
                   label=f'Expected = {fixed_spacing} ns')
        plt.xlabel('Fitted Interval [ns]', loc='right')
        plt.title(f'Bunch spacings for {id_name}')
        plt.legend(frameon=False, fontsize = 12)
        plt.tight_layout()
        plt.savefig(SavePath + f'bunch_spacings_{id_name}.png', dpi=300,
                   bbox_inches='tight', pad_inches=.3, facecolor='w')
        plt.close()
    
    # Bunch widths
    if len(valid_sigmas) > 0:
        sigma_mean = np.mean(valid_sigmas)
        plt.figure(figsize=(5, 4))
        plt.hist(valid_sigmas, bins=20, alpha=0.7, color='C0', range=(0, 9))
        plt.axvline(x=sigma_mean, color='r', linestyle='--', linewidth=2,
                   label=f'σ mean = {sigma_mean:.2f} ns')
        plt.xlabel(r'$\sigma$ [ns]', loc='right')
        plt.title(f'Bunch widths for {id_name}')
        plt.legend(frameon=False, fontsize = 12)
        plt.tight_layout()
        plt.savefig(SavePath + f'bunch_sigma_{id_name}.png', dpi=300,
                   bbox_inches='tight', pad_inches=.3, facecolor='w')
        plt.close()
        
def superimpose_bunches(timing_data, results, fixed_spacing, id_name, SavePath,
                        fit_range_fraction=0.67):
    """
    Superimpose all bunches to measure intrinsic bunch width.
    
    Parameters:
    -----------
    timing_data : array
        Original event times
    results : dict
        Output from perform_stable_fit
    fixed_spacing : float
        Bunch spacing (18.936 ns)
    id_name : str
        Identifier for plots
    SavePath : str
        Directory for saving
    fit_range_fraction : float
        Fraction of half-spacing to use for Gaussian fit (default 0.8)
    """
    
    centered_times = []
    valid_bunch_count = 0
    
    # Shift each bunch to center at zero
    for bunch_idx, bunch_center in enumerate(results['means']):
        if np.isnan(bunch_center):
            continue
        
        # Define window around this bunch
        half_width = fixed_spacing / 2
        timings = np.array(timing_data)
        mask = (timings > bunch_center - half_width) & (timings < bunch_center + half_width)
        
        filtered_times = timings[mask]
        
        if len(filtered_times) < 2:  # Skip empty bunches
            continue
        
        # Shift to center at zero
        shifted_times = filtered_times - bunch_center
        centered_times.extend(shifted_times)
        valid_bunch_count += 1
    
    if len(centered_times) == 0:
        print("Warning: No events found for superposition!")
        return None
    
    print(f"\nSuperimposed {valid_bunch_count} bunches with {len(centered_times)} total events")
    
    # Create histogram
    half_range = fixed_spacing / 2
    ranges = (-np.floor(half_range), np.ceil(half_range))
    n_bins = int(ranges[1] - ranges[0])  # 1 ns bins
    
    plt.figure(figsize=(5, 4))
    counts, bin_edges, _ = plt.hist(centered_times, bins=n_bins, range=ranges, 
                                     alpha=1, color='navy', histtype='step', linewidth=2)
    
    # Prepare for Gaussian fit
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Fit range: use central fraction to avoid tails from adjacent bunches
    fit_half_range = half_range * fit_range_fraction
    mask = (bin_centers >= -fit_half_range) & (bin_centers <= fit_half_range)
    filtered_bin_centers = bin_centers[mask]
    filtered_counts = counts[mask]
    
    # Fit Gaussian to superimposed distribution
    try:
        initial_guess = [max(filtered_counts), 0.0, 2]  # Expect ~2 ns width
        popt, pcov = curve_fit(gaussian, filtered_bin_centers, filtered_counts, 
                               p0=initial_guess,
                               bounds=([0, -2, 0.5], [np.inf, 2, 6]))
        
        amplitude, mean, sigma = popt
        sigma_error = np.sqrt(pcov[2, 2])
        
        # Plot fit
        x_fit = np.linspace(-fit_half_range, fit_half_range, 1000)
        y_fit = gaussian(x_fit, amplitude, mean, sigma)
        plt.plot(x_fit, y_fit, 'r-', linewidth=2.5, label='Gaussian fit')
        
        # Add text annotations
        plt.text(0.97, 0.95, f'{total_POT_TOR860:.2f}' + r'$\times 10^{20}$' + ' POT', 
             horizontalalignment='right', verticalalignment='top', 
             transform=plt.gca().transAxes, fontsize=10)
        plt.text(0.97, 0.85, r'$\text{N}_{\text{events}} = $' + str(len(centered_times)), 
                horizontalalignment='right', verticalalignment='top', 
                transform=plt.gca().transAxes, fontsize=10)
        plt.text(0.97, 0.75, f'$\\sigma = {sigma:.2f} \\pm {sigma_error:.2f}$ ns', 
                horizontalalignment='right', verticalalignment='top', 
                transform=plt.gca().transAxes, fontsize=10)
        
        print(f"Superimposed bunch fit:")
        print(f"  Mean: {mean:.4f} ns")
        print(f"  Sigma: {sigma:.4f} ± {sigma_error:.4f} ns")
        print(f"  (Expected BNB sigma ~ 1.308 ns)")
        
    except RuntimeError as e:
        print(f"Failed to fit superimposed distribution: {e}")
        sigma, sigma_error, mean = np.nan, np.nan, np.nan
    
    # Formatting
    plt.xlabel('Time relative to bunch center [ns]', fontsize=16)
    plt.ylabel('Events / ns', fontsize=16)
    #plt.title(f'{id_name}', fontsize=20)
    
    ax = plt.gca()
    ax.tick_params(axis='both', direction='in')
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    
    plt.xlim([ranges[0], ranges[1]])
    plt.tight_layout()
    plt.savefig(SavePath + f'superimposed_bunches_{id_name}.png', 
                dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.show()
    
    return {
        'mean': mean,
        'sigma': sigma,
        'sigma_error': sigma_error,
        'n_events': len(centered_times),
        'n_bunches': valid_bunch_count
    }


#  '''                                              '''
# ---------------------------------------------------- #
#  '''                                              '''

# Title / label for the plots
id_name = "CC"

# We choose to select the FIRST baby bunch and neglect the final one for fitting because the skyshine cut at 1/3 spill kills our stats
# this way we boost our stats by ~3% or so (1/25 bunches)

# Step 1: Fit bunches with fixed spacing
results = perform_stable_fit(
    timing_data=cluster_time,
    id_name=id_name,
    fixed_spacing=18.936,    # fixed by MicroBooNE / BNB predictions / data / observed structure
    first_peak_guess=197,
    SavePath=plot_dir,
    n_bunches=81,
    scan_range=20
)

# Step 2: Linear fit to bunch centers (to measure actual spacing)
valid_indices = [i for i, m in enumerate(results['means']) if not np.isnan(m)]
x_data = np.array(valid_indices)
y_data = np.array([results['means'][i] for i in valid_indices])

def linear_func(x, a, b):
    return a * x + b

params, covariance = curve_fit(linear_func, x_data, y_data)
slope = params[0]
intercept = params[1]
slope_error = np.sqrt(covariance[0][0])

print(f"Measured bunch spacing: {slope:.4f} ± {slope_error:.4f} ns")
print(f"Expected spacing: 18.936 ns")
print(f"Difference: {slope - 18.936:.4f} ns")

# Plot linear fit
plt.figure(figsize=(5, 4))
plt.scatter(x_data, y_data, label='Fitted bunch centers', s=10)
plt.plot(x_data, linear_func(x_data, *params), 'r-', 
         label=f'Fit: {slope:.4f} ± {slope_error:.4f} ns')
plt.xlabel('Bunch number')
plt.ylabel('Bunch Center time [ns]')
plt.title('Linear fit to bunch spacings')
plt.legend(fontsize=12)
ax = plt.gca()
ax.tick_params(axis='both', direction='in')
ax.xaxis.set_major_locator(MultipleLocator(20))
ax.xaxis.set_minor_locator(MultipleLocator(5))
plt.tight_layout()
plt.savefig(plot_dir + f'bunch_linear_fit_spacings_{id_name}.png', 
            dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.close()

# Step 3: Superimpose bunches
superimposed = superimpose_bunches(
    timing_data=cluster_time,
    results=results,
    fixed_spacing=slope,  # Use measured spacing, not 18.936
    id_name=id_name,
    SavePath=plot_dir,
    fit_range_fraction=0.6  # Fit central 80% to avoid tails
)

print('\nDone!')

In [ ]:
# Access center fits and their uncertainties to plug into the downstream scripts

bunch_centers = results['means']              # List of fitted bunch positions
bunch_widths = results['sigmas']              # List of fitted widths
bunch_center_errors = results['mean_errors']  # List of errors on bunch positions

print('Centers\n', [float(x) for x in bunch_centers], '\n\n')
print('Errors\n', [float(x) for x in bunch_center_errors], '\n')